# Healthcare Analytics Project
# Predicting 30-Day Hospital Readmission for Diabetes Patients

## Problem Statement
Hospital readmissions within 30 days increase healthcare costs and indicate potential gaps in patient care.
The objective of this project is to predict whether a diabetic patient will be readmitted within 30 days of discharge using historical hospital encounter data.

 # Dataset Description

- Source: UCI Machine Learning Repository  
- Dataset Name: Diabetes 130-US hospitals (1999–2008)  
- Number of Records: ~101,766  
- Number of Features: 50  
- Target Variable: readmitted

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [2]:
df = pd.read_csv(r"C:\Users\anupr\Desktop\Healthcare-Readmission-Prediction\Data\diabetic_data.csv")
df.shape

(101766, 50)

In [3]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [5]:
df['readmitted'].unique()

array(['NO', '>30', '<30'], dtype=object)

## Target Variable Transformation

In [6]:
# Convert the readmitted column to binary
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
df['readmitted_binary'].value_counts()

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

In [7]:
df['readmitted_binary'].value_counts(normalize=True) * 100

readmitted_binary
0    88.840084
1    11.159916
Name: proportion, dtype: float64

In [8]:
df.shape

(101766, 51)

### Observations:

- Only 11.16% of patients were readmitted within 30 days.
- The dataset is highly imbalanced.
- A model predicting only the majority class could achieve ~88% accuracy without actually identifying high-risk patients.
- Therefore, evaluation metrics like Recall and F1-score will be more important than Accuracy.

In [9]:
df.replace("?", np.nan, inplace=True)

In [10]:
df.isnull().sum().sort_values(ascending=False)

weight                      98569
max_glu_serum               96420
A1Cresult                   84748
medical_specialty           49949
payer_code                  40256
race                         2273
diag_3                       1423
diag_2                        358
diag_1                         21
troglitazone                    0
encounter_id                    0
tolazamide                      0
acarbose                        0
rosiglitazone                   0
pioglitazone                    0
tolbutamide                     0
miglitol                        0
citoglipton                     0
examide                         0
glipizide                       0
insulin                         0
glyburide-metformin             0
glipizide-metformin             0
glimepiride-pioglitazone        0
metformin-rosiglitazone         0
metformin-pioglitazone          0
change                          0
diabetesMed                     0
readmitted                      0
glyburide     

### Dropping Columns with Excessive Missing Values
Columns with more than 80% missing data were removed to ensure model reliability.

In [11]:
df.drop(columns=['weight', 'max_glu_serum', 'A1Cresult'], inplace=True)
df.shape

(101766, 48)

In [12]:
df.drop(columns=['medical_specialty', 'payer_code'], inplace=True)
df.shape 

(101766, 46)

### Removing Identifier Columns
Identifier columns such as encounter_id and patient_nbr do not contribute to prediction and were removed here.

In [13]:
df.drop(columns=['encounter_id', 'patient_nbr'], inplace=True)
df.shape

(101766, 44)

In [14]:
df['race'].fillna('Unknown', inplace=True)

In [15]:
df['diag_1'].fillna('Unknown', inplace=True)
df['diag_2'].fillna('Unknown', inplace=True)
df['diag_3'].fillna('Unknown', inplace=True)

In [16]:
df.isnull().sum().sort_values(ascending=False).head(10)

race             0
gender           0
glyburide        0
tolbutamide      0
pioglitazone     0
rosiglitazone    0
acarbose         0
miglitol         0
troglitazone     0
tolazamide       0
dtype: int64

In [17]:
df.dtypes

race                        object
gender                      object
age                         object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride                 object
acetohexamide               object
glipizide                   object
glyburide                   object
tolbutamide                 object
pioglitazone                object
rosiglitazone               object
acarbose            

In [18]:
categorical_cols = df.select_dtypes(include='object').columns
numerical_cols = df.select_dtypes(exclude='object').columns

print("Categorical Columns:", len(categorical_cols))
print("Numerical Columns:", len(numerical_cols))

Categorical Columns: 32
Numerical Columns: 12


In [19]:
df.drop(columns=['readmitted'], inplace=True)
df.shape

(101766, 43)

In [21]:
categorical_cols = df.select_dtypes(include='object').columns

In [22]:
for col in categorical_cols:
    print(col, ":", df[col].nunique())

race : 6
gender : 3
age : 10
diag_1 : 717
diag_2 : 749
diag_3 : 790
metformin : 4
repaglinide : 4
nateglinide : 4
chlorpropamide : 4
glimepiride : 4
acetohexamide : 2
glipizide : 4
glyburide : 4
tolbutamide : 2
pioglitazone : 4
rosiglitazone : 4
acarbose : 4
miglitol : 4
troglitazone : 2
tolazamide : 3
examide : 1
citoglipton : 1
insulin : 4
glyburide-metformin : 4
glipizide-metformin : 2
glimepiride-pioglitazone : 2
metformin-rosiglitazone : 2
metformin-pioglitazone : 2
change : 2
diabetesMed : 2


In [23]:
def categorize_diagnosis(diag):
    try:
        diag = float(diag)
        if 390 <= diag <= 459:
            return "Circulatory"
        elif 460 <= diag <= 519:
            return "Respiratory"
        elif 520 <= diag <= 579:
            return "Digestive"
        elif diag == 250:
            return "Diabetes"
        elif 800 <= diag <= 999:
            return "Injury"
        else:
            return "Other"
    except:
        return "Other"

In [24]:
df['diag1_cat'] = df['diag_1'].apply(categorize_diagnosis)
df['diag2_cat'] = df['diag_2'].apply(categorize_diagnosis)
df['diag3_cat'] = df['diag_3'].apply(categorize_diagnosis)

In [25]:
df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True)
df.shape

(101766, 43)

In [26]:
X = df.drop(columns=['readmitted_binary'])
y = df['readmitted_binary']

X.shape

(101766, 42)

In [27]:
X = pd.get_dummies(X, drop_first=True)
X.shape

(101766, 92)

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (81412, 92)
Testing set: (20354, 92)


Logistic Regression

In [29]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [30]:
y_pred = model.predict(X_test)

In [31]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8884248796305394

Confusion Matrix:
 [[18052    31]
 [ 2240    31]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.50      0.01      0.03      2271

    accuracy                           0.89     20354
   macro avg       0.69      0.51      0.48     20354
weighted avg       0.85      0.89      0.84     20354



In [32]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred_bal = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_bal))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bal))
print("\nClassification Report:\n", classification_report(y_test, y_pred_bal))

Accuracy: 0.660214208509384

Confusion Matrix:
 [[12241  5842]
 [ 1074  1197]]

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.68      0.78     18083
           1       0.17      0.53      0.26      2271

    accuracy                           0.66     20354
   macro avg       0.54      0.60      0.52     20354
weighted avg       0.84      0.66      0.72     20354



In [33]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Accuracy: 0.8885231404146605

Confusion Matrix:
 [[18070    13]
 [ 2256    15]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.54      0.01      0.01      2271

    accuracy                           0.89     20354
   macro avg       0.71      0.50      0.48     20354
weighted avg       0.85      0.89      0.84     20354



In [34]:
from sklearn.metrics import roc_auc_score, roc_curve
y_prob = model.predict_proba(X_test)[:,1]
auc_score = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", auc_score)

ROC-AUC Score: 0.6468526056023338


In [35]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=False)
feature_importance.head(10)

,Feature,Coefficient
25,age_[80-90),0.308954
24,age_[70-80),0.299865
9,number_inpatient,0.286478
53,rosiglitazone_No,0.262860
54,rosiglitazone_Steady,0.237244
23,age_[60-70),0.227497
26,age_[90-100),0.207619
19,age_[20-30),0.198307
20,age_[30-40),0.183237
76,diabetesMed_Yes,0.181089


In [36]:
feature_importance.tail(10)

,Feature,Coefficient
51,pioglitazone_Steady,-0.254357
81,diag1_cat_Respiratory,-0.254479
43,glipizide_No,-0.256446
28,metformin_Steady,-0.280215
40,glimepiride_Steady,-0.291444
37,chlorpropamide_Steady,-0.310144
18,age_[10-20),-0.310918
57,acarbose_Steady,-0.369339
29,metformin_Up,-0.505989
77,diag1_cat_Diabetes,-0.693354


## Final Conclusion

This project aimed to predict 30-day hospital readmissions for diabetic patients using historical hospital encounter data.

The dataset was highly imbalanced, with only ~11% of patients readmitted within 30 days. A standard Logistic Regression model achieved high accuracy but failed to detect high-risk patients effectively.

To address this, class balancing was applied, significantly improving recall for high-risk patients (from 1% to 53%). The final model achieved a ROC-AUC score of approximately 0.65, demonstrating moderate predictive capability.

Key Risk Factors Identified:
- Older age groups (70+ years)
- Higher number of prior inpatient visits
- Certain medication patterns

Protective Factors:
- Younger age groups
- Controlled diabetes management patterns

Although the model is not perfect, it provides meaningful risk stratification and demonstrates how machine learning can assist hospitals in identifying high-risk diabetic patients for early intervention and preventive care planning.